# S0.1 — CMIP6 SSP Download Manifest / SSP下载清单

This notebook generates per-scenario download manifests for SSP126, SSP245, and SSP585, reusing the model/member/variable selections from the historical pipeline (caseA/S0.1).

No family deduplication, scoring, or member selection is performed — those decisions were made in the historical S0.1 and carry over directly to SSP scenarios.

**中文说明：** 本Notebook生成SSP126/245/585三个情景的下载清单，直接复用历史管线（caseA/S0.1）的模型/member/变量选择。不重新做家族去重、打分或member选择——这些在历史S0.1中已决定，SSP直接沿用。

## 1. Configuration & historical data loading / 配置与历史数据加载

**中文说明：** 加载历史S0.1的选择结果（模型、member、变量清单），定义SSP情景参数。

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_case_dir() -> Path:
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if d.name == "caseA":
            return d
        nested = d / "case" / "caseA"
        if nested.is_dir():
            return nested
    raise FileNotFoundError("Could not locate case/caseA")


CASE_DIR = locate_case_dir()
S01_HIST_DIR = CASE_DIR / "output" / "S0.1"

FUTURE_DIR = Path.cwd().resolve()
if FUTURE_DIR.name != "case_future2609":
    for d in [FUTURE_DIR, *FUTURE_DIR.parents]:
        nested = d / "case" / "case_future2609"
        if nested.is_dir():
            FUTURE_DIR = nested
            break

OUT_DIR = FUTURE_DIR / "output" / "S0.1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SSP_ACTIVITY_ID = "ScenarioMIP"
SSP_START_YEAR = 2015
SSP_END_YEAR = 2100

print("Historical S0.1:", S01_HIST_DIR)
print("Output:", OUT_DIR)
print(f"Scenarios: {SCENARIOS}")
print(f"Period: {SSP_START_YEAR}–{SSP_END_YEAR}")

Historical S0.1: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S0.1
Output: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609/output/S0.1
Scenarios: ['ssp126', 'ssp245', 'ssp585']
Period: 2015–2100


In [2]:
hist_selected = pd.read_csv(S01_HIST_DIR / "S0.1_selected_models.csv")
hist_manifest = pd.read_csv(S01_HIST_DIR / "S0.1_download_available.csv")

print(f"Historical selected models: {len(hist_selected)}")
print(f"Historical downloadable records: {len(hist_manifest)}")
print()
display(hist_selected[["family", "model", "member_id", "pool",
                        "n_target_available", "n_target_missing"]])

Historical selected models: 35
Historical downloadable records: 629



,family,model,member_id,pool,n_target_available,n_target_missing
0,CESM2,CESM2,r1i1p1f1,core,18,0
1,CNRM,CNRM-CM6-1,r1i1p1f2,core,18,0
2,CanESM,CanESM5,r1i1p1f1,core,18,0
3,E3SM,E3SM-1-0,r1i1p1f1,core,18,0
4,GFDL,GFDL-CM4,r1i1p1f1,core,18,0
5,IPSL-CM6A,IPSL-CM6A-LR,r1i1p1f1,core,18,0
6,MIROC-ES2,MIROC-ES2H,r1i1p4f2,core,18,0
7,MRI,MRI-ESM2-0,r1i2p1f1,core,18,0
8,SAM0,SAM0-UNICON,r1i1p1f1,core,18,0
9,TaiESM,TaiESM1,r1i1p1f1,core,18,0


## 2. Generate SSP download manifests / 生成SSP下载清单

For each scenario × model × variable from the historical manifest:
- **Time variables** (core + context): same model/member/variable/grid, different experiment_id and activity_id
- **Fixed fields**: excluded — reuse from historical downloads at `/Volumes/mimi-T9/CMIP6/{model}/historical/{member}/{grid}/raw/`

**中文说明：** 基于历史清单，为每个SSP情景生成下载清单。时间变量沿用相同的model/member/variable/grid，只改experiment_id和activity_id。固定场不重新下载，复用历史已下载的。

In [3]:
hist_time = hist_manifest[hist_manifest["category"] != "fixed"].copy()

ssp_rows = []
for scenario in SCENARIOS:
    for _, row in hist_time.iterrows():
        ssp_rows.append({
            "scenario": scenario,
            "activity_id": SSP_ACTIVITY_ID,
            "model": row["model"],
            "family": row["family"],
            "member_id": row["member_id"],
            "variable": row["variable"],
            "table_id": row["table_id"],
            "grid_label": row["grid_label"],
            "category": row["category"],
            "status": "planned",
        })

ssp_manifest = pd.DataFrame(ssp_rows)

print(f"=== SSP download manifest ===")
print(f"  Scenarios: {SCENARIOS}")
print(f"  Models: {ssp_manifest['model'].nunique()}")
print(f"  Total records: {len(ssp_manifest)}")
print()

per_scenario = ssp_manifest.groupby("scenario").agg(
    n_models=("model", "nunique"),
    n_records=("variable", "count"),
    n_core=("category", lambda x: (x == "core").sum()),
    n_context=("category", lambda x: (x == "context").sum()),
).reset_index()
display(per_scenario)

print()
per_model = ssp_manifest.groupby(["scenario", "model"]).agg(
    n_vars=("variable", "count"),
).reset_index().pivot(index="model", columns="scenario", values="n_vars").fillna(0).astype(int)
print(f"Variables per model per scenario:")
display(per_model)

=== SSP download manifest ===
  Scenarios: ['ssp126', 'ssp245', 'ssp585']
  Models: 35
  Total records: 1746



,scenario,n_models,n_records,n_core,n_context
0,ssp126,35,582,103,479
1,ssp245,35,582,103,479
2,ssp585,35,582,103,479



Variables per model per scenario:


scenario,ssp126,ssp245,ssp585
model,,,
ACCESS-CM2,16,16,16
ACCESS-ESM1-5,17,17,17
AWI-ESM-1-1-LR,17,17,17
BCC-CSM2-MR,18,18,18
CAMS-CSM1-0,17,17,17
CAS-ESM2-0,17,17,17
CESM2,18,18,18
CIESM,14,14,14
CMCC-CM2-SR5,17,17,17


In [4]:
ssp_manifest.to_csv(OUT_DIR / "S0.1_ssp_download_manifest.csv", index=False)

for scenario in SCENARIOS:
    sub = ssp_manifest[ssp_manifest["scenario"] == scenario]
    sub.to_csv(OUT_DIR / f"S0.1_ssp_download_{scenario}.csv", index=False)

hist_selected.to_csv(OUT_DIR / "S0.1_selected_models.csv", index=False)

print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.glob("S0.1_*.csv")):
    print(" ", f.name)
print()
print(f"Next: S1 reads these manifests and queries ESGF for file-level records.")

Saved to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609/output/S0.1
  S0.1_selected_models.csv
  S0.1_ssp_download_manifest.csv
  S0.1_ssp_download_ssp126.csv
  S0.1_ssp_download_ssp245.csv
  S0.1_ssp_download_ssp585.csv

Next: S1 reads these manifests and queries ESGF for file-level records.
